# Run All Models Against New Benchmarks on RunPod

This notebook bootstraps a fresh RunPod Jupyter workspace, uses persistent caches when a network volume is mounted, discovers registered benchmarks with no prior repository results, and runs a resumable model/benchmark matrix.

The default `small-local` profile is intended to verify a single-GPU pod. Set `RUN_PROFILE = "all"` only on hardware capable of loading the largest 32B/72B checkpoints. OpenAI profiles require `OPENAI_API_KEY` and incur API usage.

## 1. Bootstrap the RunPod workspace

This cell finds an existing clone or clones the repository into persistent storage. It also configures Hugging Face caches and installs missing requirements. Run it before every other cell.

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/samhatescoding/transformers.git"
REPO_BRANCH = None  # Example: "main". None uses the remote default branch.

persistent_root = Path("/runpod-volume") if Path("/runpod-volume").is_dir() else Path("/workspace")
persistent_root.mkdir(parents=True, exist_ok=True)

candidate_roots = list(Path.cwd().resolve().parents) + [Path.cwd(), Path.cwd() / "transformers", persistent_root / "transformers"]
REPO_ROOT = next((path.resolve() for path in candidate_roots if (path / "models").is_dir()), None)
if REPO_ROOT is None:
    REPO_ROOT = (persistent_root / "transformers").resolve()
    clone_command = ["git", "clone"]
    if REPO_BRANCH:
        clone_command += ["--branch", REPO_BRANCH, "--single-branch"]
    clone_command += [REPO_URL, str(REPO_ROOT)]
    subprocess.run(clone_command, check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

cache_root = persistent_root / "model-cache"
cache_env = {
    "HF_HOME": cache_root / "huggingface",
    "HF_HUB_CACHE": cache_root / "huggingface" / "hub",
    "HF_DATASETS_CACHE": cache_root / "huggingface" / "datasets",
    "TRANSFORMERS_CACHE": cache_root / "huggingface" / "transformers",
    "TORCH_HOME": cache_root / "torch",
}
for name, path in cache_env.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(path)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

required_modules = ["accelerate", "datasets", "h5py", "matplotlib", "nltk", "openai", "pandas", "PIL", "psutil", "sentencepiece", "torch", "transformers"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
print(f"Installing/verifying repository requirements. Missing before install: {missing_modules or 'none'}")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
# Keep the pod's CUDA-matched torch build, but update fast-moving libraries used by newer wrappers.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "transformers", "accelerate", "datasets", "openai"],
    check=True,
)

RUN_OUTPUT_ROOT = persistent_root / "benchmark-runs"
RESULTS_DIR = RUN_OUTPUT_ROOT / "results"
SUMMARY_PATH = RUN_OUTPUT_ROOT / "all_models_new_benchmarks_summary.json"
REFERENCE_RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Persistent root: {persistent_root}")
print(f"Model cache: {cache_root}")
print(f"Run output: {RUN_OUTPUT_ROOT}")

## 2. Credentials

Add secrets in the RunPod environment before starting the pod when possible. You may set them here for the current kernel, but do not commit secret values into the notebook.

In [ ]:
# Uncomment only for an interactive session; never save real keys in this file.
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["HF_TOKEN"] = "..."  # Needed for gated Hugging Face checkpoints.

print("OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "not set")
print("HF_TOKEN:", "set" if os.getenv("HF_TOKEN") else "not set")

## 3. GPU and storage preflight

In [ ]:
import torch

free_bytes = shutil.disk_usage(persistent_root).free
print(f"Free persistent storage: {free_bytes / 2**30:.1f} GiB")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {props.name}, {props.total_memory / 2**30:.1f} GiB VRAM")
else:
    print("WARNING: local model profiles require a CUDA GPU. API-only models can run without one.")

if free_bytes < 50 * 2**30:
    print("WARNING: less than 50 GiB is free. Increase the volume before downloading several local models.")

## 4. Configure the run

Profiles:

- `small-local`: open 2B-3B local checkpoints suitable for an initial single-GPU run.
- `api`: OpenAI wrappers only; requires `OPENAI_API_KEY`.
- `all`: every public wrapper, including hardware-heavy 32B/72B models.
- `custom`: exactly the names in `CUSTOM_MODEL_NAMES`.

`NUM_SAMPLES = 2` is a smoke test. Increase it after the first successful run.

In [ ]:
RUN_PROFILE = "small-local"
CUSTOM_MODEL_NAMES = []
SELECTED_BENCHMARK_NAMES = None  # None selects every newly discovered benchmark.

NUM_SAMPLES = 2
STREAMING = True
OVERWRITE = False

## 5. Register model wrappers

In [ ]:
from models import (
    Falcon, Gemma, Gemma3_4B, Gemma3_12B, Gemma3_27B,
    Gemma4E2B, Gemma4E4B, Gemma4_26BA4B, Gemma4_31B,
    GPT4, GPT41, GPT5, GPT51, GPT52, GPT53ChatLatest,
    GPT54, GPT54Mini, GPT54Nano, GPT55, O3,
    InternVL25, InternVL25_4B, InternVL3_2B, InternVL3_8B,
    InternVL35_8BInstruct, Llama3LlavaNext8B, Llava,
    Llava15_13B, LlavaGemma2B, LlavaNextMistral7B,
    LlavaNextVicuna13B, LlavaOnevision, LlavaOnevision15_4BInstruct,
    LlavaOnevisionQwen2_7B, MiniCPMo26, MiniCPMV26,
    MiniCPMV46, MiniCPMV46Thinking, PaliGemma2_3BMix448,
    PaliGemma2_10BMix448, PaliGemma3BMix448, Qwen25VL3B,
    Qwen25VL7B, Qwen25VL32B, Qwen25VL72B, Qwen3VL4B,
    Qwen3VL8B, Qwen35_4B, Qwen35_9B,
)

MODEL_CLASSES = {
    "gpt-4o": GPT4, "gpt-4.1": GPT41, "gpt-5": GPT5,
    "gpt-5.1": GPT51, "gpt-5.2": GPT52,
    "gpt-5.3-chat-latest": GPT53ChatLatest, "gpt-5.4": GPT54,
    "gpt-5.4-mini": GPT54Mini, "gpt-5.4-nano": GPT54Nano,
    "gpt-5.5": GPT55, "o3": O3,
    "llava-1.5-7b-hf": Llava, "llava-1.5-13b-hf": Llava15_13B,
    "llava-gemma-2b": LlavaGemma2B,
    "llava-v1.6-mistral-7b-hf": LlavaNextMistral7B,
    "llava-v1.6-vicuna-13b-hf": LlavaNextVicuna13B,
    "llama3-llava-next-8b-hf": Llama3LlavaNext8B,
    "llava-onevision-qwen2-7b-ov-hf": LlavaOnevisionQwen2_7B,
    "llava-onevision-qwen2-72b-ov-hf": LlavaOnevision,
    "llava-onevision-1.5-4b-instruct": LlavaOnevision15_4BInstruct,
    "qwen2.5-vl-3b-instruct": Qwen25VL3B,
    "qwen2.5-vl-7b-instruct": Qwen25VL7B,
    "qwen2.5-vl-32b-instruct": Qwen25VL32B,
    "qwen2.5-vl-72b-instruct": Qwen25VL72B,
    "qwen3-vl-4b-instruct": Qwen3VL4B, "qwen3-vl-8b-instruct": Qwen3VL8B,
    "qwen3.5-4b": Qwen35_4B, "qwen3.5-9b": Qwen35_9B,
    "paligemma-3b-mix-224": Gemma, "paligemma-3b-mix-448": PaliGemma3BMix448,
    "paligemma2-3b-mix-448": PaliGemma2_3BMix448,
    "paligemma2-10b-mix-448": PaliGemma2_10BMix448,
    "gemma-3-4b-it": Gemma3_4B, "gemma-3-12b-it": Gemma3_12B,
    "gemma-3-27b-it": Gemma3_27B, "gemma-4-e2b-it": Gemma4E2B,
    "gemma-4-e4b-it": Gemma4E4B, "gemma-4-26b-a4b-it": Gemma4_26BA4B,
    "gemma-4-31b-it": Gemma4_31B,
    "internvl2.5-4b": InternVL25_4B, "internvl2.5-8b": InternVL25,
    "internvl3-2b": InternVL3_2B, "internvl3-8b": InternVL3_8B,
    "internvl3.5-8b-instruct": InternVL35_8BInstruct,
    "minicpm-v-2.6": MiniCPMV26, "minicpm-o-2.6": MiniCPMo26,
    "minicpm-v-4.6": MiniCPMV46,
    "minicpm-v-4.6-thinking": MiniCPMV46Thinking,
    "falcon-11b-vlm": Falcon,
}

MODEL_PROFILES = {
    "small-local": [
        "llava-gemma-2b",
        "qwen2.5-vl-3b-instruct",
    ],
    "api": [
        "gpt-4o", "gpt-4.1", "gpt-5", "gpt-5.1", "gpt-5.2",
        "gpt-5.3-chat-latest", "gpt-5.4", "gpt-5.4-mini",
        "gpt-5.4-nano", "gpt-5.5", "o3",
    ],
    "all": list(MODEL_CLASSES),
    "custom": CUSTOM_MODEL_NAMES,
}

def make_factory(model_cls):
    return lambda cls=model_cls: cls(max_new_tokens=16)

MODEL_FACTORIES = {name: make_factory(cls) for name, cls in MODEL_CLASSES.items()}
print(f"Registered {len(MODEL_FACTORIES)} model wrappers.")

## 6. Discover new, untested benchmarks

A benchmark is considered new when the repository's reference results contain no result ending in `_<benchmark>.json`. RunPod output files do not remove a benchmark from this set; they skip only the model/benchmark pairs already completed on this volume.

In [ ]:
from runners.full_suite import FULL_BENCHMARK_CLASSES

def has_any_result(benchmark_name):
    pattern = f"*_{benchmark_name}.json"
    return any(REFERENCE_RESULTS_DIR.glob(pattern))

UNTESTED_BENCHMARK_CLASSES = [
    cls for cls in FULL_BENCHMARK_CLASSES
    if not has_any_result(str(cls.benchmark_name))
]
untested_names = [str(cls.benchmark_name) for cls in UNTESTED_BENCHMARK_CLASSES]
print(f"Discovered {len(untested_names)} new benchmarks:")
print("\n".join(f"- {name}" for name in untested_names) or "- none")

## 7. Validate and preview the matrix

In [ ]:
if RUN_PROFILE not in MODEL_PROFILES:
    raise ValueError(f"Unknown RUN_PROFILE {RUN_PROFILE!r}; choose from {sorted(MODEL_PROFILES)}")

selected_model_names = MODEL_PROFILES[RUN_PROFILE]
unknown_models = sorted(set(selected_model_names) - set(MODEL_FACTORIES))
if unknown_models:
    raise ValueError(f"Unknown model names: {unknown_models}")
if not selected_model_names:
    raise ValueError("The selected model profile is empty.")

selected_benchmark_names = SELECTED_BENCHMARK_NAMES or untested_names
unknown_benchmarks = sorted(set(selected_benchmark_names) - set(untested_names))
if unknown_benchmarks:
    raise ValueError(f"Benchmarks are not in the new/untested set: {unknown_benchmarks}")

selected_benchmark_classes = [
    cls for cls in UNTESTED_BENCHMARK_CLASSES
    if str(cls.benchmark_name) in selected_benchmark_names
]
selected_factories = {name: MODEL_FACTORIES[name] for name in selected_model_names}

uses_api = any(name.startswith("gpt-") or name == "o3" for name in selected_model_names)
uses_local = any(not (name.startswith("gpt-") or name == "o3") for name in selected_model_names)
if uses_api and not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("This profile includes OpenAI models but OPENAI_API_KEY is not set.")
if uses_local and not torch.cuda.is_available():
    raise RuntimeError("This profile includes local models but CUDA is unavailable.")

print(f"Profile: {RUN_PROFILE}")
print(f"Models ({len(selected_model_names)}): {selected_model_names}")
print(f"Benchmarks ({len(selected_benchmark_classes)}): {[cls.benchmark_name for cls in selected_benchmark_classes]}")
print(f"Pairs: {len(selected_factories) * len(selected_benchmark_classes)}")
print(f"Samples per pair: {NUM_SAMPLES}")
print(f"Overwrite existing pair results: {OVERWRITE}")
if RUN_PROFILE == "all":
    print("WARNING: the all profile includes 32B/72B checkpoints and may require multi-GPU hardware plus hundreds of GiB of storage.")

## 8. Run the matrix

The runner loads one model at a time, skips completed model/benchmark files, writes each successful result immediately, records errors after every pair, releases the model, and clears the CUDA cache.

In [ ]:
from collections import Counter
import runners.full_suite as full_suite

if not selected_benchmark_classes:
    summary = []
    print("No new benchmarks remain; no models were loaded.")
else:
    original_benchmark_classes = full_suite.FULL_BENCHMARK_CLASSES
    try:
        full_suite.FULL_BENCHMARK_CLASSES = selected_benchmark_classes
        summary = full_suite.run_full_suite(
            model_factories=selected_factories,
            output_dir=RESULTS_DIR,
            summary_path=SUMMARY_PATH,
            num_samples=NUM_SAMPLES,
            overwrite=OVERWRITE,
            streaming=STREAMING,
        )
    finally:
        full_suite.FULL_BENCHMARK_CLASSES = original_benchmark_classes

print(Counter(item["status"] for item in summary))
print(f"Summary: {SUMMARY_PATH}")
print(f"Results: {RESULTS_DIR}")

## 9. Inspect results and failures

In [ ]:
import json
import pandas as pd

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else []
summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
if summary_df.empty or "status" not in summary_df:
    failures_df = pd.DataFrame()
else:
    failures_df = summary_df[~summary_df["status"].isin(["ok", "skipped"])]
failures_df